# WESAD tuned baselines: nested LOSO, three seeds, online learning curve

This notebook compares four models: Majority Dummy, Logistic Regression, RBF SVM, and Hoeffding Adaptive Tree (HAT/ADWIN). It tunes parameters using only the training participants inside each outer Leave-One-Subject-Out (LOSO) fold, then evaluates on the untouched held-out participant.

**Important interpretation:** Logistic Regression and SVM in scikit-learn use convergence-based solvers; they do not expose meaningful epoch-by-epoch metrics. Their inner-validation tuning scores and selected hyperparameters are recorded. HAT is a one-pass streaming learner, not trained for repeated epochs: its held-out stream is reported as cumulative metrics after each block of 10 chronological windows.

For HAT, after each test prediction the recorded label is revealed and used for an online update. This is replay-based test-time adaptation, and must be reported separately from the fixed batch baselines. WESAD itself has no interface interaction or usability labels.

In [ ]:
# Runtime imports first. Do not reinstall NumPy or scikit-learn.
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import LeaveOneGroupOut
print('NumPy', np.__version__, '| pandas', pd.__version__, '| scikit-learn', sklearn.__version__)

In [ ]:
# Repair/install only SciPy and River, without changing the runtime's NumPy/sklearn.
# After this cell completes, choose Runtime > Restart session, then continue below.
%pip install -q --no-cache-dir --force-reinstall --no-deps scipy==1.16.1 river==0.22.0

After restarting the runtime, continue with the import check. If imports still fail in a fresh Colab VM, select a Python 3.12 runtime before continuing.

In [ ]:
import scipy, river
from river import preprocessing as river_preprocessing
from river import tree
print('SciPy', scipy.__version__, '| River', river.__version__, '| imports OK')

## 1. Mount Drive and load the corrected participant files

This loads only `S<number>_windows.csv` files and excludes the stale combined `wesad_eda_windows.csv`.

In [ ]:
from google.colab import drive
from pathlib import Path
import re
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/DataCon/Project 6 Wesad')
files = sorted(p for p in DATA_DIR.glob('S*_windows.csv') if re.fullmatch(r'S\d+_windows', p.stem))
if len(files) != 15:
    raise ValueError(f'Expected 15 corrected subject CSVs, found {len(files)} in {DATA_DIR}')
df = pd.concat((pd.read_csv(p) for p in files), ignore_index=True)
print('Files:', len(files), '| rows:', len(df), '| participants:', df.subject.nunique())
display(df.head())

## 2. Audit and define predictors

The IDs and time index are excluded from predictors. `label_id` is also excluded because it directly encodes the target label.

In [ ]:
required = {'subject','window','label','label_id'}
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'
feature_cols = [c for c in df.columns if c not in required]
assert feature_cols, 'No sensor features found'
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors='coerce')
assert df[feature_cols].notna().all().all(), 'Missing features: only load corrected participant CSVs'
assert np.isfinite(df[feature_cols].to_numpy(float)).all(), 'Non-finite features found'
assert df.duplicated().sum() == 0, 'Duplicate rows found; check Drive folder for duplicate file versions'
assert df.groupby('subject').window.apply(lambda s: s.is_monotonic_increasing).all(), 'Rows are not chronological within subjects'
assert len(df) == 3281 and df.subject.nunique() == 15, f'Unexpected dataset size: {df.shape}, subjects={df.subject.nunique()}'
CLASS_ORDER = ['baseline','stress','amusement']
assert set(df.label.unique()) == set(CLASS_ORDER), f'Unexpected classes: {sorted(df.label.unique())}'
X = df[feature_cols].astype(float).reset_index(drop=True)
y = df.label.astype(str).reset_index(drop=True)
groups = df.subject.astype(str).reset_index(drop=True)
df = df.reset_index(drop=True)
print('Class counts:', y.value_counts().to_dict())
print('Quality checks passed.')

## 3. Nested participant-grouped tuning and outer LOSO

Outer LOSO produces one test fold per person. Inside each outer training set, three GroupShuffleSplit validation folds are used to select parameters; no outer test participant is used for tuning, imputation, or scaling. Three seeds vary inner participant splits and HAT randomness. The same outer participant is tested under every seed.

**Search spaces:** LR C = 0.01, 0.1, 1, 10, 100 with class weights none/balanced. SVM C = 0.1, 1, 10, 100 and gamma = scale, 0.01, 0.1 with class weights none/balanced. HAT grace period = 100/200 and delta = 1e-5/1e-7. Majority Dummy has no tunable parameters. All selection uses mean inner-validation macro-F1. The grid is intentionally compact; it can be expanded after runtime and leakage checks work.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from itertools import product
import json

SEEDS = [17, 42, 2026]
OUTER = LeaveOneGroupOut()
LR_GRID = list(product([0.01, 0.1, 1.0, 10.0, 100.0], [None, 'balanced']))
SVM_GRID = list(product([0.1, 1.0, 10.0, 100.0], ['scale', 0.01, 0.1], [None, 'balanced']))
HAT_GRID = list(product([100, 200], [1e-5, 1e-7]))

def metrics(y_true, y_pred):
    return {'accuracy': accuracy_score(y_true, y_pred),
            'macro_f1': f1_score(y_true, y_pred, labels=CLASS_ORDER, average='macro', zero_division=0),
            'macro_precision': precision_score(y_true, y_pred, labels=CLASS_ORDER, average='macro', zero_division=0),
            'macro_recall': recall_score(y_true, y_pred, labels=CLASS_ORDER, average='macro', zero_division=0)}

def batch_pipeline(kind, params, seed):
    if kind == 'Logistic Regression':
        C, class_weight = params
        clf = LogisticRegression(C=C, class_weight=class_weight, max_iter=3000, solver='lbfgs', random_state=seed)
    elif kind == 'RBF SVM':
        C, gamma, class_weight = params
        clf = SVC(C=C, gamma=gamma, kernel='rbf', class_weight=class_weight)
    else:
        raise ValueError(kind)
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), clf)

def stream_rows(frame, seed, shuffle_subjects=False):
    subjects = np.array(sorted(frame.subject.astype(str).unique()))
    if shuffle_subjects:
        np.random.default_rng(seed).shuffle(subjects)
    for subject in subjects:
        part = frame[frame.subject.astype(str) == subject].sort_values('window')
        for _, row in part.iterrows():
            xrow = {col: float(row[col]) for col in feature_cols}
            yield xrow, str(row.label), subject, int(row.window)

def fit_hat(train_frame, params, seed):
    grace_period, delta = params
    model = river_preprocessing.StandardScaler() | tree.HoeffdingAdaptiveTreeClassifier(
        grace_period=grace_period, delta=delta, seed=seed, bootstrap_sampling=True)
    for xrow, label, _, _ in stream_rows(train_frame, seed, shuffle_subjects=True):
        model.learn_one(xrow, label)
    return model

def evaluate_hat_replay(model, test_frame, chunk_size=10):
    actuals, predictions, curve = [], [], []
    ordered = test_frame.sort_values('window')
    for i, (_, row) in enumerate(ordered.iterrows(), start=1):
        xrow = {col: float(row[col]) for col in feature_cols}
        true_label = str(row.label)
        pred = model.predict_one(xrow)  # predict before revealing this label
        pred = str(pred) if pred is not None else CLASS_ORDER[0]
        actuals.append(true_label)
        predictions.append(pred)
        model.learn_one(xrow, true_label)  # replay feedback arrives after prediction
        if i % chunk_size == 0 or i == len(ordered):
            m = metrics(actuals, predictions)
            curve.append({'windows_seen': i, 'block_size': chunk_size, **m})
    return actuals, predictions, curve

def tune_batch(kind, train_idx, seed):
    params_grid = LR_GRID if kind == 'Logistic Regression' else SVM_GRID
    inner = GroupShuffleSplit(n_splits=3, test_size=0.25, random_state=seed)
    Xtr, ytr, gtr = X.iloc[train_idx], y.iloc[train_idx], groups.iloc[train_idx]
    scores = []
    for params in params_grid:
        fold_scores = []
        for fit_local, val_local in inner.split(Xtr, ytr, groups=gtr):
            model = batch_pipeline(kind, params, seed)
            model.fit(Xtr.iloc[fit_local], ytr.iloc[fit_local])
            pred = model.predict(Xtr.iloc[val_local])
            fold_scores.append(f1_score(ytr.iloc[val_local], pred, labels=CLASS_ORDER, average='macro', zero_division=0))
        scores.append((float(np.mean(fold_scores)), params))
    return max(scores, key=lambda item: item[0])

def tune_hat(train_idx, seed):
    train_frame = df.iloc[train_idx].copy()
    inner = GroupShuffleSplit(n_splits=3, test_size=0.25, random_state=seed)
    scores = []
    for params in HAT_GRID:
        fold_scores = []
        for fit_local, val_local in inner.split(train_frame, groups=train_frame.subject.astype(str)):
            fit_frame = train_frame.iloc[fit_local]
            val_frame = train_frame.iloc[val_local]
            model = fit_hat(fit_frame, params, seed)
            actual, pred, _ = evaluate_hat_replay(model, val_frame, chunk_size=1000000)
            fold_scores.append(f1_score(actual, pred, labels=CLASS_ORDER, average='macro', zero_division=0))
        scores.append((float(np.mean(fold_scores)), params))
    return max(scores, key=lambda item: item[0])

## 4. Run the tuned experiments

This can take several minutes because tuning is repeated within every outer participant fold. HAT is initialized using the training participants and then adapted sequentially on the held-out person's stream. Its curve reports cumulative performance by chronological window blocks; these blocks are **not training epochs**.

In [ ]:
records, curves, chosen = [], [], []
for seed in SEEDS:
    print('Seed:', seed, flush=True)
    for fold, (train_idx, test_idx) in enumerate(OUTER.split(X, y, groups=groups), start=1):
        train_subjects = set(groups.iloc[train_idx]); test_subjects = set(groups.iloc[test_idx])
        assert len(test_subjects) == 1 and train_subjects.isdisjoint(test_subjects), 'Outer subject leakage'
        held_out = next(iter(test_subjects))
        train_frame, test_frame = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

        # Majority-class reference: no tuning.
        dummy = make_pipeline(SimpleImputer(strategy='median'), DummyClassifier(strategy='prior'))
        dummy.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = dummy.predict(X.iloc[test_idx])
        records.append({'seed':seed,'fold':fold,'held_out_subject':held_out,'model':'Majority Dummy',
                        'best_inner_macro_f1':np.nan,'best_params':'{}','n_test_windows':len(test_idx),**metrics(y.iloc[test_idx],pred)})

        for kind in ['Logistic Regression', 'RBF SVM']:
            inner_score, params = tune_batch(kind, train_idx, seed)
            model = batch_pipeline(kind, params, seed)
            model.fit(X.iloc[train_idx], y.iloc[train_idx])
            pred = model.predict(X.iloc[test_idx])
            records.append({'seed':seed,'fold':fold,'held_out_subject':held_out,'model':kind,
                            'best_inner_macro_f1':inner_score,'best_params':json.dumps(params),
                            'n_test_windows':len(test_idx),**metrics(y.iloc[test_idx],pred)})
            chosen.append({'seed':seed,'fold':fold,'held_out_subject':held_out,'model':kind,
                           'best_inner_macro_f1':inner_score,'best_params':json.dumps(params)})

        inner_score, params = tune_hat(train_idx, seed)
        hat = fit_hat(train_frame, params, seed)
        actual, pred, curve = evaluate_hat_replay(hat, test_frame, chunk_size=10)
        records.append({'seed':seed,'fold':fold,'held_out_subject':held_out,
                        'model':'Hoeffding Adaptive Tree (ADWIN)','best_inner_macro_f1':inner_score,
                        'best_params':json.dumps({'grace_period':params[0],'delta':params[1]}),
                        'n_test_windows':len(actual),**metrics(actual,pred)})
        chosen.append({'seed':seed,'fold':fold,'held_out_subject':held_out,'model':'Hoeffding Adaptive Tree (ADWIN)',
                       'best_inner_macro_f1':inner_score,'best_params':json.dumps({'grace_period':params[0],'delta':params[1]})})
        for point in curve:
            curves.append({'seed':seed,'fold':fold,'held_out_subject':held_out,'model':'Hoeffding Adaptive Tree (ADWIN)',**point})
        print(f'  fold {fold}/15 held out {held_out}: HAT inner macro-F1={inner_score:.3f}; test macro-F1={records[-1]['macro_f1']:.3f}',flush=True)

results = pd.DataFrame(records)
learning_curve = pd.DataFrame(curves)
best_parameters = pd.DataFrame(chosen)
print('Evaluations:',len(results),'expected:',4*15*3)
display(results.head())

## 5. Summarize and report maxima correctly

Average the three seed results within each held-out participant first. Then report the mean and standard deviation across the 15 participants. Also show the best and worst participant scores as ranges; do not call the single best held-out score the model's expected performance. Hyperparameters are selected only by inner validation macro-F1, never by outer test scores.

In [ ]:
score_cols=['accuracy','macro_f1','macro_precision','macro_recall']
per_subject=(results.groupby(['model','held_out_subject'],as_index=False)[score_cols].mean())
summary=per_subject.groupby('model')[score_cols].agg(['mean','std','min','max']).round(3)
display(summary)
print('Selected hyperparameter counts:')
display(best_parameters.groupby(['model','best_params']).size().rename('fold_seed_count').reset_index().sort_values(['model','fold_seed_count'],ascending=[True,False]))
print('HAT chronological learning curve, averaged across folds/seeds by windows seen:')
hat_curve=(learning_curve.groupby('windows_seen')[['accuracy','macro_f1']].mean().round(3))
display(hat_curve.head(30))

## 6. Save results to Drive

In [ ]:
OUT_DIR=DATA_DIR/'tuned_baseline_results'
OUT_DIR.mkdir(exist_ok=True)
results.to_csv(OUT_DIR/'wesad_nested_loso_tuned_raw.csv',index=False)
per_subject.to_csv(OUT_DIR/'wesad_nested_loso_tuned_per_subject.csv',index=False)
summary.to_csv(OUT_DIR/'wesad_nested_loso_tuned_summary.csv')
best_parameters.to_csv(OUT_DIR/'wesad_nested_loso_best_parameters.csv',index=False)
learning_curve.to_csv(OUT_DIR/'wesad_hat_prequential_learning_curve.csv',index=False)
print('Saved outputs to',OUT_DIR)

## Limits to state in your report

- This is nested, participant-held-out evaluation on only 15 lab participants. Report uncertainty and avoid broad real-world claims.
- HAT's held-out stream labels are revealed after prediction for replay adaptation. This differs from a system with no immediate labels. Do not compare it as though it were a fixed classifier without also reporting its adaptation protocol.
- Ten-window cumulative blocks are learning-curve checkpoints, not training epochs. Batch Logistic Regression and SVM have no epoch values in this setup; the notebook reports their inner tuning score and chosen parameters instead.
- Verify sensor-label synchronization against the original WESAD release before final publication.
- WESAD has no HCI interaction outcomes. A separate user study is required to show that interface adaptation helps.